# 05 — Preprocess and train a real model, step by step

**Plain-language question:** How can training and future prediction apply
exactly the same cleanup?

**Why this matters:** if preprocessing is learned separately or repeated by
hand, training and inference can transform the same row differently.

**Estimated time:** 60–75 minutes.
**Prerequisite:** lessons 00–04; you know missing values, train/validation, a
confusion matrix, and the no-skill baseline.


## Preflight

Check the kernel, then load the two development splits.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
import numpy as np

from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.evaluation import evaluate_probabilities
from aai_local_classification.modeling import feature_frame
from aai_local_classification.workflow import ensure_prepared

ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
validation = load_split(settings, SplitName.VALIDATION, paths.data_root)
X_train = feature_frame(train, settings)
X_validation = feature_frame(validation, settings)
y_train = train.churned_30d
y_validation = validation.churned_30d
print(f"✓ X_train {X_train.shape}; X_validation {X_validation.shape}")


### What you should see

Nine raw input columns for 2,160 training rows and 360 validation rows.

### Words introduced

| Word | Plain meaning | Example |
|---|---|---|
| imputation | Fill a missing value using a learned rule | training median |
| scaling | Re-express numbers on comparable scales | mean 0, spread 1 |
| one-hot encoding | Turn each category into indicator columns | plan_basic 0/1 |


## Find real missing inputs

**Before you run this:** predict whether a missing value should be replaced
using information from train alone or train plus validation.


In [ ]:
missing_examples = X_train.loc[
    X_train.isna().any(axis=1),
    ["usage_hours_30d", "signup_channel", "plan_tier"],
].head(5)
missing_examples


### How to interpret the output

`NaN` means the value is unknown; it is not the number zero or a category named
“None.” The imputer must learn replacement values from training only. Learning
them from validation would let validation influence the fitted pipeline.


## Build the numeric and categorical paths visibly

An sklearn **transformer** learns a data transformation with `fit` and applies
it with `transform`. A **Pipeline** chains steps so they are fit and applied in
the same order.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    [
        ("impute_median", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("impute_most_common", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)


`handle_unknown="ignore"` means a future category that did not occur in
training will not crash inference. It does not refit or invent evidence from the
future row.

Now combine the two paths. `remainder="drop"` excludes undeclared columns.


In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, list(settings.features.numeric)),
        (
            "categorical",
            categorical_pipeline,
            list(settings.features.categorical),
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)


## Fit on train; transform validation

**Before you run this:** nine raw columns will expand because each category gets
its own indicator. Predict whether the transformed matrix may contain missing
values.


In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_validation_encoded = preprocessor.transform(X_validation)
encoded_names = preprocessor.get_feature_names_out()

pd.Series(
    {
        "raw_columns": X_train.shape[1],
        "encoded_columns": X_train_encoded.shape[1],
        "training_missing_after_transform": int(np.isnan(X_train_encoded).sum()),
        "validation_missing_after_transform": int(np.isnan(X_validation_encoded).sum()),
    }
).to_frame("value")


### What you should see

Nine raw features become 16 numeric columns and both missing counts are zero.
The fitted categorical names must not include a fake `signup_channel_None`
category—the missing channel was genuinely imputed.


In [ ]:
encoded_preview = pd.DataFrame(X_train_encoded[:3], columns=encoded_names)
print(encoded_names.tolist())
encoded_preview


### How to interpret the output

The five numeric inputs appear once each; categories expand into 0/1 columns.
Scaled numeric values can be negative because zero now represents the training
mean—not because the original fee or tenure was negative.


## Put preprocessing and logistic regression in one artifact

### Words introduced

| Word | Plain meaning | Here |
|---|---|---|
| estimator | An object that learns from examples | logistic regression |
| `fit` | Learn preprocessing and model parameters | training rows only |
| `predict_proba` | Return a probability for each class | churn score in column 1 |

Logistic regression learns a weighted linear combination of the encoded
features. It is a useful interpretable first candidate, not proof of causality.


In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=1000, random_state=settings.random_seed),
        ),
    ]
)
logistic_pipeline.fit(X_train, y_train)
print("✓ Fitted one complete preprocessing + model Pipeline")


**Before you run this:** predict whether the first five probabilities must all
be between 0 and 1. The `classes_` lookup avoids assuming class-column order.


In [ ]:
positive_index = list(logistic_pipeline.classes_).index(1)
validation_probability = logistic_pipeline.predict_proba(X_validation)[
    :, positive_index
]
pd.DataFrame(
    {
        "true_label": y_validation.head().to_numpy(),
        "churn_probability": validation_probability[:5],
        "prediction_at_0.5": (validation_probability[:5] >= 0.5).astype(int),
    }
)


### What you should see

Five different probabilities in `[0, 1]`. At threshold 0.5, many rows remain
negative. A probability is not a promise that a particular account will churn;
it is the model's score under this fitted data and specification.


In [ ]:
metrics_at_half = evaluate_probabilities(
    y_validation,
    validation_probability,
    0.5,
    false_negative_cost=settings.selection.false_negative_cost,
    false_positive_cost=settings.selection.false_positive_cost,
)
pd.Series(
    {
        "accuracy": metrics_at_half.accuracy,
        "precision": metrics_at_half.precision,
        "recall": metrics_at_half.recall,
        "average_precision": metrics_at_half.average_precision,
    }
).to_frame("validation value")


### How to interpret the output

Accuracy is around 81%, but recall at 0.5 is only about 11%. The model ranks
positives much better than the baseline (AP around 0.46 versus 0.19), yet the
default threshold is a poor action policy. Lesson 06 separates ranking choice
from threshold choice.

### Misconception check

A Pipeline is not just notebook cells in order. It is one fitted object that
carries the learned imputation, scaling, encoding, and classifier together into
future inference.


## MLOps bridge: what must be recorded

The baseline run from lesson 04 recorded a model artifact. The controlled
comparison in lesson 06 will record this whole Pipeline, an input example,
input/output signature, parameters, metrics, dataset fingerprints, and exact
dependency evidence. Keeping preprocessing inside the artifact prevents
training/serving mismatch.


### Guided exercise

Copy one validation row, set `signup_channel` to a category never seen during
training, and ask the existing fitted Pipeline for a probability. Do not refit.


In [ ]:
exercise_future = X_validation.head(1).copy()
exercise_future.loc[:, "signup_channel"] = "brand_new_channel"
exercise_probability = logistic_pipeline.predict_proba(exercise_future)[
    0, positive_index
]
print(f"Probability for unseen category: {exercise_probability:.3f}")


**Self-check:** the prediction should succeed and remain between 0 and 1.
`handle_unknown="ignore"` reuses the training vocabulary; it does not learn the
new category.

<details><summary>Solution explanation</summary>

The complete Pipeline accepts the raw row, ignores the unknown category's
one-hot indicators, applies all other learned transformations, and scores it.
</details>


In [ ]:
# Reference solution — run after your attempt
assert 0 <= exercise_probability <= 1
assert "brand_new_channel" not in encoded_names
print("✓ Unseen category handled without refitting")


## Recap

- Imputation, scaling, and encoding learn rules from training only.
- One sklearn Pipeline carries those learned transformations with the model.
- A strong ranking score does not make 0.5 the correct action threshold.

**Evidence created:** a logistic Pipeline in this kernel's memory. No candidate
selection or test decision was persisted, so lesson 06 remains the first place
that chooses and records a candidate.

**Ready for 06?** You can explain why validation uses `transform`, not
`fit_transform`, and why the preprocessing belongs inside the model artifact.
